# Lab 4 · Before vs After

**~20 minutes.** The point of the whole day.

We reload the model with the adapter from Lab 3, ask the **exact same
twelve questions** as Lab 1, and compare.

One thing to hold onto: those twelve questions were **removed from the
training data**. The model saw paraphrases of the same facts, never these
exact strings. So this measures generalisation, not recall.

In [ ]:
# --- Install the fine-tuning stack --------------------------------------
# Kaggle ships a matched torch/CUDA pair. --no-deps stops pip replacing torch
# with an incompatible build, which shows up later as baffling CUDA errors.
#
# Takes 2-4 minutes. "dependency conflict" warnings here are expected and fine.
# Output is deliberately NOT suppressed: if this step fails, you need to see it.
!pip install -q --no-deps unsloth unsloth_zoo
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets huggingface_hub sentencepiece protobuf

print("\ninstall finished - verifying imports...")
import importlib
missing = [m for m in ("unsloth", "trl", "peft", "bitsandbytes", "datasets")
           if importlib.util.find_spec(m) is None]
print("MISSING: " + ", ".join(missing) if missing else "all packages importable")

In [ ]:
# --- Locate the workshop repo -------------------------------------------
# Tries, in order: already present -> attached Kaggle Dataset -> git clone.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Ankush610/LLM-lab.git"

def find_repo() -> Path:
    for candidate in [Path("/kaggle/working/LLM-lab"), Path.cwd(), Path.cwd().parent]:
        if (candidate / "common" / "config.py").exists():
            return candidate
    for d in Path("/kaggle/input").glob("*"):          # attached as a Dataset
        if (d / "common" / "config.py").exists():
            return d
    print("Repo not found locally, cloning...")        # last resort
    subprocess.run(["git", "clone", "-q", REPO_URL, "/kaggle/working/LLM-lab"], check=True)
    return Path("/kaggle/working/LLM-lab")

REPO = find_repo()
sys.path[:0] = [str(REPO / "common"), str(REPO / "dataset")]
print(f"repo: {REPO}")

import config
print(config.summary())

In [ ]:
# --- Hugging Face authentication ----------------------------------------
# Reads the Kaggle Secret named HF_TOKEN. Never paste a token into a cell:
# it is saved with the notebook and shared notebooks leak tokens constantly.
import os

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
    print("Hugging Face: authenticated")
except Exception as e:
    print(f"Hugging Face: NOT authenticated ({type(e).__name__})")
    print("  Fix: right panel -> Add-ons -> Secrets -> add HF_TOKEN, tick the box.")
    print("  Or set USE_UNGATED_MODEL = True below to skip the gated model entirely.")

## 5.1 Load the fine-tuned model

Loading from the adapter directory pulls the 4-bit base model and applies
our ~90 MB of LoRA weights on top.

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = str(config.ADAPTER_DIR),
    max_seq_length = config.MAX_SEQ_LENGTH,
    dtype          = None,
    load_in_4bit   = config.LOAD_IN_4BIT,
)
FastLanguageModel.for_inference(model)      # ~2x faster inference path
print("fine-tuned model ready")

In [ ]:
def ask(question, system=config.SYSTEM_PROMPT, **kw):
    msgs = ([{"role": "system", "content": system}] if system else []) + \
           [{"role": "user", "content": question}]
    inputs = tokenizer.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
    params = dict(max_new_tokens=config.MAX_NEW_TOKENS, do_sample=False,
                  pad_token_id=tokenizer.eos_token_id)
    params.update(kw)
    out = model.generate(input_ids=inputs, **params)
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

## 5.2 Ask the same twelve questions

In [ ]:
from eval_prompts import EVAL_PROMPTS
import compare

tuned = {}
for i, p in enumerate(EVAL_PROMPTS, 1):
    print(f"[{i:>2}/{len(EVAL_PROMPTS)}] {p['question'][:58]}...")
    tuned[p["id"]] = ask(p["question"])

compare.save(tuned, config.TUNED_ANSWERS)

## 5.3 Side by side

Green = the answer contains the facts we expect. Red = it doesn't.

The scoring is a crude keyword check on purpose — it makes the change
countable, but **the real evaluation is you reading the two columns.**

In [ ]:
baseline = compare.load(config.BASELINE_ANSWERS)
compare.report(baseline, tuned)

In [ ]:
# Two in full, for a proper read.
for p in EVAL_PROMPTS[:2]:
    print("=" * 74)
    print("Q:", p["question"])
    print("-" * 74)
    print("BEFORE:\n", baseline[p["id"]][:420])
    print("\nAFTER:\n", tuned[p["id"]][:420])
    print()

## 5.5 Did we break anything else?

Fine-tuning on a narrow domain can degrade general ability — *catastrophic
forgetting*. With LoRA on 665 examples for 2 epochs it should be mild,
because the base weights never moved.

Judge for yourself.

In [ ]:
from eval_prompts import OUT_OF_DOMAIN_PROMPTS

for q in OUT_OF_DOMAIN_PROMPTS:
    print(f"Q: {q}")
    print(f"A: {ask(q, max_new_tokens=110).strip()[:320]}\n")

## 5.6 The adapter is a switch

`disable_adapter()` turns the LoRA weights off without unloading anything.
Same weights in memory, two different behaviours, toggled in a line.

This is the mental model to take away: **an adapter is swappable behaviour,
not a new model.** You keep one base model and as many adapters as you have
tasks, and they cost 90 MB each.

> ↳ Slide: *What is an Adapter*

In [ ]:
q = "Which partition should I use for a 4-hour A100 job on AURA?"

print("--- adapter ON ---")
print(ask(q).strip()[:300])

with model.disable_adapter():
    print("\n--- adapter OFF (base model) ---")
    print(ask(q).strip()[:300])

## 5.7 Being honest about what just happened

The improvement is real, but let's be precise about what it is.

**What fine-tuning did well**
- Taught a consistent format, voice and level of detail
- Taught a bounded set of facts that appear many times in the data
- Taught *when to say "I don't know"* — the dataset includes refusals

**What it did not do**
- Give the model a database. 665 examples is a behaviour, not a knowledge base.
- Make it reliable on facts seen once or twice
- Remove hallucination. It hallucinates less *here*, and will still invent things off-topic.

**When you'd reach for something else**

| situation | better tool |
|---|---|
| Facts change weekly | RAG — retrieve at query time |
| Thousands of documents | RAG |
| Must cite sources | RAG |
| Need a specific format/voice/behaviour | **fine-tuning** |
| Need a domain's vocabulary and conventions | **fine-tuning** |
| Need lower latency and cost than few-shot prompting | **fine-tuning** |

In production the two are complements, not rivals: fine-tune the
behaviour, retrieve the facts.

---

### Next: `05_save_and_gguf.ipynb` — saving, merging and model formats

> **Kaggle tip:** if the session has been idle a while, check the right-hand
> panel still shows the GPU attached before starting the next notebook.